# Solutions, Lab 05: Divergence Choice, One Variable, Measured Honestly

This notebook solves the four exercises at the end of `labs/lab-05-divergence-ablation.ipynb`.
Lab 05 is a Tier 2 lab, so its six-run matrix executes only with `RUN_TRAINING = True` on a
training box. The solutions split as follows:

- **Exercise 1 (beta 0.25 and 0.75):** the mechanism, that intermediate beta is not a
  behavioral interpolation, is demonstrated live through divergence values and gradient
  directions on synthetic logits; the four extra runs are gated.
- **Exercise 2 (temperature interaction):** the mechanism, that a softened teacher gives the
  forward-KL student more tail to cover, is demonstrated live with a capacity-limited student;
  the T=2 sweep is gated.
- **Exercise 3 (where did the tail go):** the exercise's measurement, student probability on
  the teacher's second-ranked token, runs fully live on the real model pair in fp32; the
  trained-checkpoint comparison is gated.
- **Exercise 4 (TVD as a control):** the gradient's existence, boundedness, and geometry are
  verified live; the TVD training arm is gated.

Live measurements use the lab's own model pair, SmolLM2-360M-Instruct as teacher and
SmolLM2-135M-Instruct as student, in fp32 on a CPU-sized slice of the eval corpus. Attempt
the exercises before reading on; the judgment they build is the product, and these pages are
only its receipt.

In [1]:
import os
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")   # widget progress bars crash some notebook stacks; plain logs are fine

import sys, os, json, math, gc, time
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import (gjsd, kl_divergence, tvd, shift_for_next_token, top1_agreement,
                     mean_entropy, distinct_n, self_bleu, masked_mean)
from kd_pipeline import set_seed_everywhere, config_fingerprint, RunManifest

RUN_TRAINING = False        # <-- flip on the training box, same flag as the lab
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# The lab's run matrix, copied verbatim so gated solutions extend the same experiment.
BASE = dict(
    teacher="HuggingFaceTB/SmolLM2-360M-Instruct",
    student="HuggingFaceTB/SmolLM2-135M-Instruct",
    data="../data/lab03", seq_len=384, T=1.0,
    lr=3e-5, batch_size=8, grad_accum=4, max_steps=800, warmup_steps=40,
)
BETAS, SEEDS = (0.0, 0.5, 1.0), (17, 18)
MATRIX = [{**BASE, "beta": b, "seed": s} for b in BETAS for s in SEEDS]

def diff_keys(a, b):
    return {k for k in a.keys() | b.keys() if a.get(k) != b.get(k)}

print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Exercise 1: Add beta = 0.25 and beta = 0.75

**The exercise, restated.** Extend the matrix with beta = 0.25 and 0.75. Is the entropy trend
monotone in beta, or does it step at the endpoints? The lab's A.1 already warned that
intermediate beta is not a behavioral interpolation.

**The approach.** The runs are gated, but the question "what objective does beta = 0.25
actually train" is answerable now, and answering it first tells us what entropy trend to
expect. A.1 proved a statement about the *value* of the divergence: near beta = 0 the value
is beta times the forward KL. But training does not follow values, it follows gradients, so
the live cell measures gradient *directions*: for each beta I compute the gradient of
`gjsd` with respect to the student logits and its cosine similarity (the cosine of the angle
between two vectors, 1.0 meaning same direction) against the gradients of pure forward and
pure reverse KL on the same inputs. The inputs are deliberately peaked, mismatched teacher
and student logits (3 sigma scale), because that is the regime where forward and reverse KL
genuinely disagree; on nearly matched distributions all these gradients converge and the
comparison shows nothing.

One numerical footnote the grid has to respect: at exactly beta = 0 the mixture m equals the
student distribution, so `gjsd` returns identically zero with a zero gradient (and
symmetrically at beta = 1). That is why TRL special-cases the endpoints, and why my grid uses
0.001 and 0.999 as stand-ins for them while the exact endpoint objectives come from
`kl_divergence` directly.

In [2]:
set_seed_everywhere(SEED)
g = torch.Generator().manual_seed(SEED)
V = 96
zt = 3.0 * torch.randn(2, 6, V, generator=g)     # peaked teacher
zs = 3.0 * torch.randn(2, 6, V, generator=g)     # peaked student, mismatched
m = torch.ones(2, 6, dtype=torch.bool)

def grad_of(loss_fn):
    z = zs.clone().requires_grad_(True)
    loss_fn(z).backward()
    return z.grad.flatten()

cos = F.cosine_similarity
g_fwd = grad_of(lambda z: kl_divergence(z, zt, m, scale_by_T2=False))
g_rev = grad_of(lambda z: kl_divergence(z, zt, m, direction="reverse", scale_by_T2=False))
fwd = float(kl_divergence(zs, zt, m, scale_by_T2=False))
rev = float(kl_divergence(zs, zt, m, direction="reverse", scale_by_T2=False))
print(f"forward KL {fwd:.3f}, reverse KL {rev:.3f}, "
      f"cos(fwd grad, rev grad) = {float(cos(g_fwd, g_rev, dim=0)):.3f}\n")

print(f"{'beta':>6} {'gjsd':>8} {'gjsd/beta':>10} {'cos vs fwd':>11} {'cos vs rev':>11}")
table = {}
for b in (0.001, 0.25, 0.5, 0.75, 0.999):
    gb = grad_of(lambda z: gjsd(z, zt, m, beta=b))
    v = float(gjsd(zs, zt, m, beta=b))
    cf, cr = float(cos(gb, g_fwd, dim=0)), float(cos(gb, g_rev, dim=0))
    table[b] = (v, cf, cr)
    print(f"{b:>6} {v:>8.4f} {v/b:>10.3f} {cf:>11.3f} {cr:>11.3f}")

# The A.1 limit, now in gradient space: near beta=0 the objective IS forward KL
# at 1/1000 the magnitude, not a new objective.
assert table[0.001][1] > 0.85, "near beta=0 the gradient points along forward KL"
assert 0.75 < table[0.001][0] / 0.001 / fwd < 1.05, "and its value is beta * forward KL"
# The quarter points lean toward their nearer endpoint, they do not blend to something new.
assert table[0.25][1] > table[0.25][2], "beta=0.25 gradient is closer to forward KL"
assert table[0.75][2] > table[0.75][1], "beta=0.75 gradient is closer to reverse KL"
# The symmetric point is bounded by ln 2, the JSD's known maximum.
assert table[0.5][0] <= math.log(2) + 1e-6, "JSD at beta=0.5 is bounded by ln 2 = 0.693"
print("\nCHECK ex1-live: beta moves the gradient between two fixed directions (and "
      "rescales it); it does not manufacture intermediate behaviors")

forward KL 6.067, reverse KL 5.931, cos(fwd grad, rev grad) = 0.294

  beta     gjsd  gjsd/beta  cos vs fwd  cos vs rev
 0.001   0.0053      5.258       0.911       0.318
  0.25   0.4718      1.887       0.574       0.439
   0.5   0.5836      1.167       0.514       0.492
  0.75   0.4674      0.623       0.459       0.557
 0.999   0.0050      0.005       0.340       0.884

CHECK ex1-live: beta moves the gradient between two fixed directions (and rescales it); it does not manufacture intermediate behaviors


In [3]:
# Exercise 1, gated part: the extended matrix. train_one and sample_and_measure are
# the lab's Part B functions with the loss made a parameter, so exercises 2 and 4 reuse them.
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

def gjsd_loss(s_sh, t_sh, m_sh, cfg):
    return gjsd(s_sh, t_sh, m_sh, beta=cfg["beta"], T=cfg["T"])

def train_one(cfg, loss_fn=gjsd_loss):
    set_seed_everywhere(cfg["seed"])
    student = AutoModelForCausalLM.from_pretrained(cfg["student"], dtype=torch.bfloat16).to(device)
    teacher = AutoModelForCausalLM.from_pretrained(cfg["teacher"], dtype=torch.bfloat16).to(device).eval()
    for p in teacher.parameters():
        p.requires_grad_(False)
    tr = torch.load(os.path.join(cfg["data"], "train.pt"))
    batches = [{k: tr[k][i:i+cfg["batch_size"]].to(device) for k in ("input_ids", "mask")}
               for i in range(0, len(tr["input_ids"]), cfg["batch_size"])]
    opt = torch.optim.AdamW(student.parameters(), lr=cfg["lr"])
    sched = get_cosine_schedule_with_warmup(opt, cfg["warmup_steps"], cfg["max_steps"])
    step = 0
    while step < cfg["max_steps"]:
        for b in batches:
            s_logits = student(b["input_ids"]).logits
            with torch.no_grad():
                t_logits = teacher(b["input_ids"]).logits
            s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, b["mask"])
            loss = loss_fn(s_sh, t_sh, m_sh, cfg) / cfg["grad_accum"]
            loss.backward()
            if (step + 1) % cfg["grad_accum"] == 0:
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                opt.step(); sched.step(); opt.zero_grad()
            step += 1
            if step >= cfg["max_steps"]:
                break
    return student, teacher

@torch.no_grad()
def sample_and_measure(student, teacher, cfg, n_prompts=32, max_new=96):
    tok = AutoTokenizer.from_pretrained(cfg["student"])
    ev = torch.load(os.path.join(cfg["data"], "eval.pt"))
    texts = []
    for i in range(n_prompts):
        plen = ev["prompt_lens"][i]
        prompt = ev["input_ids"][i:i+1, :plen].to(device)
        out = student.generate(prompt, do_sample=True, temperature=1.0, top_p=1.0,
                               max_new_tokens=max_new, pad_token_id=tok.eos_token_id)
        texts.append(tok.decode(out[0, plen:], skip_special_tokens=True))
    ids = ev["input_ids"][:64].to(device); mm = ev["mask"][:64].to(device)
    s_logits = student(ids).logits; t_logits = teacher(ids).logits
    s_sh, t_sh, m_sh = shift_for_next_token(s_logits, t_logits, mm)
    return {"entropy": mean_entropy(s_sh, m_sh), "distinct3": distinct_n(texts, 3),
            "self_bleu": self_bleu(texts, 3),
            "agreement": top1_agreement(s_sh, t_sh, m_sh)}

EXTENDED = [{**BASE, "beta": b, "seed": s} for b in (0.25, 0.75) for s in SEEDS]
for cfg in EXTENDED:
    assert diff_keys(cfg, MATRIX[0]) <= {"beta", "seed"}, "still moving beta and seed only"
print(f"{len(EXTENDED)} additional runs verified against the matrix discipline")

if RUN_TRAINING:
    results = []
    for cfg in EXTENDED:
        student, teacher = train_one(cfg)
        row = {"beta": cfg["beta"], "seed": cfg["seed"],
               **sample_and_measure(student, teacher, cfg)}
        results.append(row); print(row)
        del student, teacher
        if device == "cuda":
            torch.cuda.empty_cache()
    with open("../runs/sol05_extended.json", "w") as f:
        json.dump(results, f, indent=2)
else:
    print("RUN_TRAINING=False: the four extra runs are written but did not execute here.")

/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4 additional runs verified against the matrix discipline
RUN_TRAINING=False: the four extra runs are written but did not execute here.


**Interpretation.** The live table answers the exercise's parenthetical before any
training does. Reading it: at beta = 0.001 the gradient's cosine against pure forward KL was
about 0.91 and the value divided by beta recovered the forward KL to within about 13 percent
(the exact limit as beta goes to 0; at 0.001 it has not fully converged on logits this
peaked), so the near-endpoint objective is forward KL with a tiny effective learning rate,
exactly as A.1 claimed. At beta = 0.25 the gradient still leaned clearly toward the forward
direction (cosine 0.57 versus 0.44 toward reverse, against a fwd-to-rev baseline cosine of
0.29), and symmetrically at 0.75; nowhere in the sweep does a genuinely new direction
appear. The picture that emerges: beta slides the gradient along a one-parameter family
between two fixed endpoint directions and simultaneously rescales it, shrinking toward zero
at both ends.

**The four extra runs did not execute in this build; expected shape, not results.** From the
mechanism: the entropy trend across beta in {0, 0.25, 0.5, 0.75, 1} should be monotone in
*ordering* but not linear in *spacing*: 0.25 lands close to the 0.0 arm and 0.75 close to
the 1.0 arm, with the visible step between 0.25 and 0.75, because each quarter-point trains
a scaled version of its nearer endpoint's objective. Scale matters too: the beta-out-front
factor means the 0.25 and 0.75 arms take smaller effective steps at the lab's fixed learning
rate, so at 800 steps they may sit even closer to their endpoint's *starting* entropy than
the endpoint arms themselves, mimicking a step function. Grade it with the lab's registered
rule: the 0.25-versus-0.0 gap must exceed the seed spread before you call it a real
difference; Part C's noise floor (entropy gaps of 0.1 to 0.5 nats across the full beta
range) says most of the range's movement should sit between 0.25 and 0.75. A smooth linear
staircase across all five points would actually *refute* the gradient analysis above, and
the first suspect would be the learning-rate confound A.1 warned about.

## Exercise 2: Temperature interaction

**The exercise, restated.** Rerun the beta sweep at T = 2. Softening the teacher gives the
forward-KL student more tail to cover. Which gaps widen?

**The approach.** The sweep is gated, but "more tail to cover" is a measurable claim about
the objective, not about training, and it is the whole causal story, so the live cell
measures it directly. The subtlety worth spelling out: for an *unconstrained* student,
temperature changes nothing, because the minimizer of every one of these divergences is
"student logits equal teacher logits" at any T. The interaction only exists because a real
student has less capacity than its teacher and must triage. So the live demonstration builds
the smallest honest model of that situation: a teacher over 512 tokens, and a student that
can only represent the teacher's top 32 tokens (its logits match the teacher's there and sit
at a floor everywhere else), standing in for a small model that covers the common tokens and
has nothing left for the rare ones. For that fixed pair I measure, at T = 1 and T = 2: the
teacher's probability mass outside the student's support (the tail the student cannot
cover), the forward KL, and the reverse KL. The prediction: softening multiplies the
uncovered tail mass several times over, the forward KL (which pays for every uncovered
teacher token) rises by more than the reverse KL does, and therefore the pressure that
separates the beta = 0 arm from the beta = 1 arm grows with T.

In [4]:
set_seed_everywhere(SEED)
V = 512
zt2 = 2.5 * torch.randn(1, 1, V)                    # the teacher
m1 = torch.ones(1, 1, dtype=torch.bool)
J = 32                                              # the student's capacity: top-32 support
top_idx = zt2.topk(J, -1).indices
zs2 = torch.full((1, 1, V), -20.0)                  # floor: effectively zero probability
zs2.scatter_(-1, top_idx, zt2.gather(-1, top_idx))  # matches the teacher on its support

print(f"{'T':>4} {'teacher H (nats)':>17} {'tail beyond support':>20} "
      f"{'forward KL':>11} {'reverse KL':>11}")
res = {}
for T in (1.0, 2.0):
    H = mean_entropy(zt2, m1, T=T)
    p = F.softmax(zt2 / T, -1)
    tail = 1.0 - float(p.gather(-1, top_idx).sum())
    f = float(kl_divergence(zs2, zt2, m1, T=T, scale_by_T2=False))
    r = float(kl_divergence(zs2, zt2, m1, T=T, direction="reverse", scale_by_T2=False))
    res[T] = (H, tail, f, r)
    print(f"{T:>4} {H:>17.3f} {tail:>20.4f} {f:>11.3f} {r:>11.3f}")

H1, tail1, f1, r1 = res[1.0]
H2, tail2, f2, r2 = res[2.0]
assert H2 > H1, "softening raises the teacher's entropy"
assert tail2 > 2 * tail1, "softening multiplies the mass the student cannot cover"
assert f2 > f1 and r2 > r1, "both divergences feel the mismatch grow"
assert (f2 - f1) > (r2 - r1), \
    "but the forward KL, which pays for every uncovered teacher token, grows more"
print(f"\nCHECK ex2-live: at T=2 the uncovered tail went from {tail1:.0%} to {tail2:.0%} "
      f"of teacher mass; forward KL rose {f2-f1:.2f} nats vs {r2-r1:.2f} for reverse")

   T  teacher H (nats)  tail beyond support  forward KL  reverse KL
 1.0             3.833               0.2004       4.231       0.224
 2.0             5.531               0.6255       5.723       0.982

CHECK ex2-live: at T=2 the uncovered tail went from 20% to 63% of teacher mass; forward KL rose 1.49 nats vs 0.76 for reverse


In [5]:
# Exercise 2, gated part: the full beta sweep rerun at T=2.
MATRIX_T2 = [{**BASE, "T": 2.0, "beta": b, "seed": s} for b in BETAS for s in SEEDS]
for cfg in MATRIX_T2:
    assert diff_keys(cfg, MATRIX[0]) <= {"T", "beta", "seed"}
print(f"{len(MATRIX_T2)} runs at T=2, moving T once and beta x seed within it")

if RUN_TRAINING:
    results = []
    for cfg in MATRIX_T2:
        student, teacher = train_one(cfg)
        row = {"T": cfg["T"], "beta": cfg["beta"], "seed": cfg["seed"],
               **sample_and_measure(student, teacher, cfg)}
        results.append(row); print(row)
        del student, teacher
        if device == "cuda":
            torch.cuda.empty_cache()
    with open("../runs/sol05_T2.json", "w") as f:
        json.dump(results, f, indent=2)
else:
    print("RUN_TRAINING=False: the T=2 sweep is written but did not execute here.")

6 runs at T=2, moving T once and beta x seed within it
RUN_TRAINING=False: the T=2 sweep is written but did not execute here.


**Interpretation.** The live table isolates the mechanism to three numbers per
temperature. Softening the teacher from T=1 to T=2 raised its entropy by more than a nat and
multiplied the mass beyond the capacity-limited student's support roughly threefold (the
printed check), and the two divergences priced that growth differently: the forward KL rose
by about twice as many nats as the reverse KL. The asymmetry is the entire temperature
interaction. Forward KL charges the student for every teacher token it fails to cover, so
tripling the uncoverable mass triples the pressure to smear; reverse KL charges the student
only where the *student* puts mass, and the student's support is inside the teacher's
high-probability region at any temperature, so it barely notices.

**The T=2 sweep did not execute in this build; expected shape, not results.** The gaps that
widen are the ones driven by mode covering: the entropy gap between beta = 0 and beta = 1
should stretch beyond its T=1 value (Part C's 0.1 to 0.5 nats band moves toward and past its
upper end), and distinct-3 with it, because the beta = 0 student is now dragged even further
toward smearing while beta = 1 stays anchored on the modes. Self-BLEU widens mirror-wise.
The agreement gap is the one I would *not* expect to widen cleanly: both arms' agreement is
measured on argmax tokens, which softening barely reorders, and the beta = 0 arm's extra
smearing costs it little on top-1. If instead every metric widens by a similar factor,
suspect the gradient-scale confound rather than the tail story: `gjsd` carries no T-squared
compensation, so check the raw loss magnitudes at T=2 before crediting the geometry.

## Exercise 3: Where did the tail go?

**The exercise, restated.** For the beta = 0 and beta = 1 students, plot the per-position
student probability of the teacher's *second*-ranked token. Reverse KL's tail-dropping should
be visible directly, not just through entropy.

**The approach.** The exercise names the single best measurement in this lab: the teacher's
rank-2 token is the first token that mode-seeking is allowed to abandon (rank 1 is the mode
it keeps) and the last one mode-covering is allowed to drop, so the student's probability on
it is where the two objectives disagree most visibly. No trained Lab 05 checkpoints exist on
this build machine, so the live cell does what can be done honestly: run the measurement
itself, end to end, on the real pair the lab trains, the 360M teacher and the *untrained*
135M student, fp32, over a 16-row, 192-position slice of the eval set. That produces the
baseline curve both trained students start from, and it proves the machinery (rank
extraction, masked gathering, the summary statistics) on real tensors. The gated cell applies
the identical function to the beta = 0 and beta = 1 checkpoints once Part B has produced
them. For scale, I also record the teacher's own probability on its rank-2 token, which is
the mode-covering ideal: a student that matched the teacher perfectly would sit exactly
there.

In [6]:
from transformers import AutoModelForCausalLM
from transformers.utils import logging as hf_logging
hf_logging.disable_progress_bar()

set_seed_everywhere(SEED)
T_SUB, N_ROWS, BS = 192, 16, 4
ev = torch.load("../data/lab03/eval.pt")
good = (ev["mask"][:, :T_SUB].sum(1) >= 40).nonzero().squeeze(-1)
rows = good[:N_ROWS]
ids_sub = ev["input_ids"][rows][:, :T_SUB]
mask_sh = ev["mask"][rows][:, 1:T_SUB]              # mask shifted once, with the logits

@torch.no_grad()
def rank2_probability(student_name_or_dir, teacher_name):
    # Returns per-position student probability of the teacher's rank-2 token,
    # plus the teacher's own probability of it, over supervised positions.
    teacher = AutoModelForCausalLM.from_pretrained(teacher_name, dtype=torch.float32).eval()
    student = AutoModelForCausalLM.from_pretrained(student_name_or_dir, dtype=torch.float32).eval()
    s_r2, t_r2 = [], []
    for i in range(0, N_ROWS, BS):
        t_log = teacher(ids_sub[i:i+BS]).logits[:, :-1]
        s_log = student(ids_sub[i:i+BS]).logits[:, :-1]
        top2 = t_log.topk(2, dim=-1).indices[..., 1:]           # the rank-2 token id
        t_p = F.softmax(t_log, -1).gather(-1, top2).squeeze(-1)
        s_p = F.softmax(s_log, -1).gather(-1, top2).squeeze(-1)
        msk = mask_sh[i:i+BS]
        s_r2.append(s_p[msk]); t_r2.append(t_p[msk])
    del teacher, student; gc.collect()
    return torch.cat(s_r2), torch.cat(t_r2)

t0 = time.time()
s_r2, t_r2 = rank2_probability(BASE["student"], BASE["teacher"])
q = lambda x, p: float(x.quantile(p))
print(f"{len(s_r2)} supervised positions measured in {time.time()-t0:.0f}s\n")
print(f"{'':<26} {'mean':>8} {'median':>8} {'25th pct':>9} {'75th pct':>9}")
print(f"{'teacher p(rank-2 token)':<26} {float(t_r2.mean()):>8.4f} {q(t_r2,0.5):>8.4f} "
      f"{q(t_r2,0.25):>9.4f} {q(t_r2,0.75):>9.4f}")
print(f"{'student p(rank-2 token)':<26} {float(s_r2.mean()):>8.4f} {q(s_r2,0.5):>8.4f} "
      f"{q(s_r2,0.25):>9.4f} {q(s_r2,0.75):>9.4f}")
ratio = float(s_r2.mean() / t_r2.mean())
frac_low = float((s_r2 < 0.01).float().mean())
print(f"\nstudent/teacher mean ratio: {ratio:.2f}; fraction of positions where the "
      f"student gives the token under 1% probability: {frac_low:.1%}")

assert len(s_r2) > 800, "enough positions for stable quantiles"
assert torch.isfinite(s_r2).all() and float(s_r2.min()) >= 0 and float(s_r2.max()) <= 1
assert torch.isfinite(t_r2).all()
print("CHECK ex3-live: the tail measurement runs end to end on the real pair; the numbers "
      "above are the untrained baseline both beta arms start from")

1175 supervised positions measured in 26s

                               mean   median  25th pct  75th pct
teacher p(rank-2 token)      0.1049   0.0518    0.0028    0.1850
student p(rank-2 token)      0.1139   0.0387    0.0032    0.1647

student/teacher mean ratio: 1.09; fraction of positions where the student gives the token under 1% probability: 35.6%
CHECK ex3-live: the tail measurement runs end to end on the real pair; the numbers above are the untrained baseline both beta arms start from


In [7]:
# Exercise 3, gated part: the same measurement on the trained beta=0 and beta=1
# students, once Lab 05's Part B has written checkpoints (extend Part B to save them,
# e.g. student.save_pretrained(f"../runs/lab05/beta{beta}_seed{seed}")).
if RUN_TRAINING and os.path.isdir("../runs/lab05"):
    ckpts = [d for d in sorted(os.listdir("../runs/lab05"))
             if os.path.isdir(os.path.join("../runs/lab05", d))]
    for ck in ckpts:
        s_p, t_p = rank2_probability(os.path.join("../runs/lab05", ck), BASE["teacher"])
        print(f"{ck}: student mean p(rank-2) {float(s_p.mean()):.4f}  "
              f"median {float(s_p.quantile(0.5)):.4f}  "
              f"under-1% fraction {float((s_p < 0.01).float().mean()):.1%}")
else:
    print("Gated: rerun with RUN_TRAINING=True after Part B has saved beta=0 and beta=1 "
          "checkpoints under ../runs/lab05; this cell then prints the trained comparison.")

Gated: rerun with RUN_TRAINING=True after Part B has saved beta=0 and beta=1 checkpoints under ../runs/lab05; this cell then prints the trained comparison.


**Interpretation.** Reading the live baseline, and it holds a small surprise worth
being honest about: the untrained student's *mean* probability on the teacher's rank-2
token (0.114) actually sits at the teacher's own level (0.105, ratio 1.09), so "the small
model starts below the teacher on near-miss tokens" would be the wrong summary. The medians
tell the truer story: the student's median (0.039) runs below the teacher's (0.052), and
over a third of positions already get under 1 percent student probability, which means both
distributions are heavily skewed and the student's mean is propped up by a minority of
positions where it happens to pile mass on that token, sometimes because it disagrees about
rank 1 entirely. That skew is the reason this exercise asks for the per-position view: a
mean, like mean entropy, hides exactly the shape that mode-seeking will change.

**The trained comparison did not execute in this build; expected shape, not results.** Both
trained arms start from the baseline row above. The beta = 0 (forward KL, mode covering)
student should tighten toward the teacher's row *position by position*: median rising toward
the teacher's, quartiles narrowing toward the teacher's, and the under-1-percent fraction
falling, because covering the teacher means matching it on precisely these near-miss tokens.
The beta = 1 (reverse KL, mode seeking) student should move the other way: median drifting
down from the baseline and the under-1-percent fraction growing past its starting 36
percent, because every unit of mass reclaimed from rank-2 tokens is reinvested in rank-1
modes the objective actually rewards. The
distance between the two arms on this measurement should exceed what Part C's entropy gap
(0.1 to 0.5 nats) implies, since this statistic concentrates the whole effect on the tokens
where it lives. Failure signature: if the beta = 1 arm's rank-2 probability *rises* along
with its entropy falling, the mass is being taken from somewhere deeper in the tail than
rank 2, which happens with very peaked teachers; extend the measurement to rank 3 through 10
(one line: widen the `topk`) before concluding the theory failed.

## Exercise 4: TVD as a control

**The exercise, restated.** Add a `tvd`-trained arm. Total variation distance is bounded like
JSD but is not an f-divergence interpolation between the two KLs. Where does it land on the
metric table, and does that match its geometry?

**The approach.** Before spending a training run on a loss, verify it can train at all: that
its gradient exists, is finite, and is bounded (TVD contains an absolute value, which is not
differentiable at zero, so this deserves an explicit check rather than faith), and then read
its geometry from the gradients themselves. Three live measurements: (1) value and gradient sanity, including the [0, 1] bound; (2) direction: cosine of
the TVD gradient against the forward-KL and reverse-KL gradients, where the geometric
expectation is near-symmetry, since TVD's integrand |p - q| treats the two distributions
identically; (3) the property that actually distinguishes TVD in training, gradient
*saturation*: once the two distributions barely overlap, |p - q| is at its ceiling almost
everywhere, so the gradient through it collapses toward zero and the objective stops
pushing. I measure that by comparing gradient norms on the moderate pair against a severely
mismatched pair. The training arm is gated, and the placement prediction below is a worked
answer, one defensible reading of the geometry, not the answer.

One design note on the inputs: parts (1) and (2) use a *moderately* overlapping pair of
distributions (unit-scale logits), not exercise 1's 3-sigma peaked pair. The reason is
measurement hygiene: on the peaked pair TVD already sits above 0.9, most positions are
saturated, and the surviving gradient comes from the few overlap tokens whose accidents
dominate the direction. Saturation belongs in part (3), measured on purpose, not smuggled
into part (2) as noise.

In [8]:
set_seed_everywhere(SEED)
# A moderately overlapping pair: unit-scale logits, where all the divergences
# still have room to disagree without saturating.
gm = torch.Generator().manual_seed(SEED)
zs_mod = torch.randn(2, 6, V, generator=gm)
zt_mod = torch.randn(2, 6, V, generator=gm)

def grad_on_mod(loss_fn):
    z = zs_mod.clone().requires_grad_(True)
    loss_fn(z).backward()
    return z.grad.flatten()

# (1) value and gradient sanity.
z = zs_mod.clone().requires_grad_(True)
v = tvd(z, zt_mod, m)
v.backward()
vv = float(v.detach())
assert 0.0 <= vv <= 1.0, "TVD is bounded in [0, 1]"
assert torch.isfinite(z.grad).all(), "the gradient exists and is finite despite |.|"
assert float(z.grad.norm()) > 0, "and it is not degenerate"
g_tvd = z.grad.flatten()
print(f"tvd = {vv:.4f}, grad max |entry| = {float(z.grad.abs().max()):.2e}, "
      f"grad norm = {float(z.grad.norm()):.2e}")

# (2) geometry: TVD's gradient sits symmetrically between the two KL directions.
g_fwd_m = grad_on_mod(lambda zz: kl_divergence(zz, zt_mod, m, scale_by_T2=False))
g_rev_m = grad_on_mod(lambda zz: kl_divergence(zz, zt_mod, m, direction="reverse",
                                               scale_by_T2=False))
c_f = float(cos(g_tvd, g_fwd_m, dim=0))
c_r = float(cos(g_tvd, g_rev_m, dim=0))
assert abs(c_f - c_r) < 0.2, "near-symmetric pull toward both KL directions"
print(f"cos(tvd grad, fwd-KL grad) = {c_f:.3f}, cos(tvd grad, rev-KL grad) = {c_r:.3f}")

# (3) saturation: when the distributions barely overlap, the gradient dies.
# Exercise 1's 3-sigma pair is already most of the way there; a 12-sigma pair is
# essentially disjoint.
v_peaked = float(tvd(zs, zt, m))
g2 = torch.Generator().manual_seed(SEED + 1)
zt_far = 12.0 * torch.randn(2, 6, V, generator=g2)      # near-disjoint peaked pair
z_far = (12.0 * torch.randn(2, 6, V, generator=g2)).requires_grad_(True)
v_far = tvd(z_far, zt_far, m)
v_far.backward()
norm_mod, norm_far = float(g_tvd.norm()), float(z_far.grad.norm())
print(f"moderate mismatch:  tvd {vv:.3f}, grad norm {norm_mod:.2e}")
print(f"peaked ex-1 pair:   tvd {v_peaked:.3f} (already near the ceiling)")
print(f"severe mismatch:    tvd {float(v_far.detach()):.3f}, grad norm {norm_far:.2e}")
assert float(v_far.detach()) > 0.95, "near-disjoint distributions saturate the bound"
assert norm_far < 0.2 * norm_mod, "and the gradient collapses with them"
print("\nCHECK ex4-live: tvd is trainable (finite, bounded, symmetric gradient) but its "
      "signal dies exactly where the student is worst; predict placement accordingly")

tvd = 0.5055, grad max |entry| = 8.67e-04, grad norm = 6.56e-03
cos(tvd grad, fwd-KL grad) = 0.737, cos(tvd grad, rev-KL grad) = 0.690
moderate mismatch:  tvd 0.506, grad norm 6.56e-03
peaked ex-1 pair:   tvd 0.914 (already near the ceiling)
severe mismatch:    tvd 0.979, grad norm 8.12e-07

CHECK ex4-live: tvd is trainable (finite, bounded, symmetric gradient) but its signal dies exactly where the student is worst; predict placement accordingly


In [9]:
# Exercise 4, gated part: the TVD arm, two seeds, same matrix discipline.
def tvd_loss(s_sh, t_sh, m_sh, cfg):
    return tvd(s_sh, t_sh, m_sh, T=cfg["T"])

TVD_ARMS = [{**BASE, "beta": "tvd", "seed": s} for s in SEEDS]
for cfg in TVD_ARMS:
    assert diff_keys(cfg, MATRIX[0]) <= {"beta", "seed"}, "the loss is the only new variable"
print("TVD arm configs verified against the matrix discipline")

if RUN_TRAINING:
    results = []
    for cfg in TVD_ARMS:
        student, teacher = train_one(cfg, loss_fn=tvd_loss)
        row = {"loss": "tvd", "seed": cfg["seed"],
               **sample_and_measure(student, teacher, cfg)}
        results.append(row); print(row)
        del student, teacher
        if device == "cuda":
            torch.cuda.empty_cache()
    with open("../runs/sol05_tvd.json", "w") as f:
        json.dump(results, f, indent=2)
else:
    print("RUN_TRAINING=False: the TVD arm is written but did not execute here.")

TVD arm configs verified against the matrix discipline
RUN_TRAINING=False: the TVD arm is written but did not execute here.


**Interpretation.** The three live checks give TVD a character sketch. It is
trainable: finite bounded values, a finite gradient everywhere PyTorch evaluates it (the
absolute value's kink at zero is handled by autograd's subgradient convention, and the
measured entries are small and bounded). It is symmetric in practice, not just on paper: its
gradient's cosine to the forward-KL direction and to the reverse-KL direction came out
within five hundredths of each other (0.74 versus 0.69), so unlike any beta setting of the
JSD family, TVD has no meaningful mode-covering or mode-seeking lean. And it saturates: on
the near-disjoint pair the value pinned near its ceiling while the gradient norm fell by
almost four orders of magnitude versus the moderate pair (6.6e-3 down to 8.1e-7), with
exercise 1's peaked pair already at 0.91 of the ceiling, which means TVD pushes hardest when student and teacher
already mostly agree and goes quiet exactly where the student is worst, the reverse of
forward KL's temperament.

**The placement prediction, as one defensible answer, not the answer** (the arm is gated and
did not execute here; grade it against your own table). On entropy, distinct-3, and
self-BLEU, TVD should land nearest the beta = 0.5 JSD row: bounded, symmetric objectives
sit together on the smear-versus-concentrate axis. On top-1 agreement it should land at or
slightly below JSD, and the saturation measurement is why: early in training, when the
student is far from the teacher at many positions, TVD's gradient there is weakest, so at
the lab's fixed 800-step budget the TVD student simply moves less, which reads on the table
as metrics closest of all arms to the untrained baseline. That is the sense in which it is a
*control*: it marks where "a bounded loss that trains gently" lands, so any JSD-versus-KL
difference smaller than the JSD-versus-TVD difference should not be attributed to mode
geometry. Failure signature: a TVD arm that matches reverse KL's low entropy would suggest
entropy collapse from some other cause (learning rate, masking), not mode seeking, because
the live geometry shows TVD has no seeking preference to express.